# jpomop walkthrough with invented data

This notebook runs jpomop from start to finish on 100 invented patients.
It shows the two input files, how they are checked and converted, and the eight OMOP CDM tables that come out.

Every patient, hospital, drug code and procedure code here is invented.
The ICD-10 codes are real, but the patients and their records are invented.
The notebook does not need any hospital data, code master or Athena download.

jpomop is at an early stage (version 0.0.1) and has not been run on real hospital data.
This notebook shows what the code does with files that follow the specification, not how well it works on real files.

To run it:

```sh
git clone https://github.com/kojikojikojikoji/jpomop
cd jpomop
pip install -e . pandas jupyter
jupyter lab examples/walkthrough_en.ipynb
```

A Japanese version is in `walkthrough_ja.ipynb`.

## 1. Set up

All files go into a temporary folder, and the last cell deletes it.

In [1]:
from pathlib import Path
import tempfile

import duckdb
import pandas as pd

import jpomop

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

SEED = 31                # see section 2
FOLLOW = "0000000048"    # the patient we follow (data_id in the DPC files)
FOLLOW_PERSON = 48       # the same patient in the CDM tables

work = Path(tempfile.mkdtemp(prefix="jpomop_"))
print("jpomop", jpomop.__version__)

jpomop 0.0.1


## 2. Make the invented data

`generate` writes two files for June 2026.
The same seed always gives the same files, byte for byte.

- `yoshiki1.txt` is Form 1 (様式1). It has the patient, the admission, the discharge, the diagnoses and the surgeries of each stay.
- `ef.txt` is the EF file (入院EF統合ファイル). It has billed items and their service dates.

Both files are tab-separated text in Shift JIS, as the DPC manual requires. jpomop reads them as CP932, the Windows form of Shift JIS.

This notebook uses seed 31.
With this seed, one patient (`data_id` 0000000048) has every case that the notebook explains in a single stay: a ward move, a surgery, a drug brought from home, a drug given at discharge and a diagnosis that arose after admission.
The default seed of `jpomop synth` gives other files.

In [2]:
from jpomop.synth import generate

m = generate(work / "synth", n_patients=100, seed=SEED)
print(m.n_patients, "patients,", m.n_stays, "stays")
for p in (m.yoshiki1, m.ef):
    print(p.name, p.stat().st_size, "bytes,", len(p.read_bytes().splitlines()), "lines")

100 patients, 120 stays
yoshiki1.txt 73350 bytes, 900 lines
ef.txt 337757 bytes, 1840 lines


## 3. What Form 1 looks like

Form 1 is a long file, with several lines for each stay.
Each line has 17 fields: eight fixed fields, then nine payload fields.
The `code` field says what the payload fields mean on that line.

Here are the lines for patient 0000000048.
The rest of this notebook follows this patient.

In [3]:
from jpomop.spec import load_spec

y1_spec = load_spec("dpc_yoshiki1")
lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()
raw_y1 = pd.DataFrame([ln.split("\t") for ln in lines], columns=y1_spec.field_names())
raw_y1[raw_y1["data_id"] == FOLLOW]

,facility_code,data_id,admission_date,stay_no,summary_no,code,version,seq,payload1,payload2,payload3,payload4,payload5,payload6,payload7,payload8,payload9
413,999999999,0000000048,20260605,0,0,A000010,20140401,0,19420906,1,1000001,,,,,,
414,999999999,0000000048,20260605,0,0,A000020,20140401,0,20260605,1,,,100,0,,,
415,999999999,0000000048,20260605,0,0,A000030,20140401,0,20260610,1,4,0,,,,,
416,999999999,0000000048,20260605,0,0,A006010,20140401,0,,C189,,9000006,,,,,結腸癌
417,999999999,0000000048,20260605,0,0,A006020,20140401,0,,C189,,9000006,,,,,結腸癌
418,999999999,0000000048,20260605,0,0,A006030,20140401,0,,C189,,9000006,,,,,結腸癌
419,999999999,0000000048,20260605,0,0,A006040,20140401,1,,E785,,9000103,,,,,脂質異常症
420,999999999,0000000048,20260605,0,0,A006040,20140401,2,,N189,,9000104,,,,,慢性腎臓病
421,999999999,0000000048,20260605,0,0,A007010,20140401,1,20260606,K9999,,,,,,,架空手術
422,999999999,0000000048,20260605,0,0,A006050,20140401,1,,J690,,9000201,,,,,誤嚥性肺炎


How to read these lines:

| `code` | What the line holds | Payload fields used above |
|---|---|---|
| A000010 | The patient | 1: birth date, 2: sex, 3: postcode |
| A000020 | The admission | 1: admission date, 2: where the patient came from |
| A000030 | The discharge | 1: discharge date, 2: where the patient went, 3: outcome, 4: death within 24 hours |
| A006010 | The main diagnosis | 2: ICD-10, 4: disease code, 5: modifier code, 9: name |
| A006020 | The diagnosis that led to the admission | same as A006010 |
| A006030 | The diagnosis that used the most resources | same as A006010 |
| A006040 | A diagnosis the patient already had at admission | same as A006010 |
| A006050 | A diagnosis that arose after admission | same as A006010 |
| A007010 | A surgery | 1: date, 2: code in the fee schedule, 9: name |

This patient was in hospital from 5 June to 10 June 2026.
The main diagnosis is C189 (colon cancer), and the same code is on three lines.
E785 and N189 were present at admission.
J690 (aspiration pneumonia) arose after admission.
There was one surgery on 6 June.

The modifier field (payload 5) is empty here.
When it holds 8002, the diagnosis is "suspected". Section 8 shows an example.

`summary_no` is 統括診療情報番号 in the manual.
The value 0 means that the form covers the whole stay.
Section 10 shows a form with another value.

jpomop keeps the layout in a YAML file.
Each field has its position, type, length and the page of the manual where it is defined.

In [4]:
pd.DataFrame(
    [(f.col, f.name, f.name_ja, f.type, f.length, f.page) for f in y1_spec.fields],
    columns=["col", "name", "name_ja", "type", "length", "manual_page"],
).astype({"length": "Int64"})

,col,name,name_ja,type,length,manual_page
0,1,facility_code,施設コード,digits,9,22
1,2,data_id,データ識別番号,digits,10,22
2,3,admission_date,入院年月日,date8,8,22
3,4,stay_no,回数管理番号,string,<NA>,22
4,5,summary_no,統括診療情報番号,string,<NA>,22
5,6,code,コード,string,7,22
6,7,version,バージョン,date8,8,22
7,8,seq,連番,integer,<NA>,22
8,9,payload1,ペイロード1,string,<NA>,22
9,10,payload2,ペイロード2,string,<NA>,22


## 4. What the EF file looks like

The EF file has 31 fields on each line.
Each line records a billed item and its service date: a drug, a test, a procedure or a room charge.
The table below shows eight of the 31 fields for the same patient, sorted by date.

In [5]:
ef_spec = load_spec("dpc_ef")
lines = m.ef.read_bytes().decode("cp932").splitlines()
raw_ef = pd.DataFrame([ln.split("\t") for ln in lines], columns=ef_spec.field_names())
cols = ["exec_date", "data_kubun", "receipt_code", "item_name", "quantity",
        "detail_flags", "act_points", "ward_code"]
raw_ef.loc[raw_ef["data_id"] == FOLLOW, cols].sort_values("exec_date", kind="stable")

,exec_date,data_kubun,receipt_code,item_name,quantity,detail_flags,act_points,ward_code
887,20260605,90,190999901,架空入院基本料,0000000.000,000000000000,100,W1
888,20260605,70,170999901,架空画像診断,0000000.000,000000000000,100,W1
902,20260605,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,002200000000,0,W1
889,20260606,90,190999901,架空入院基本料,0000000.000,000000000000,100,W1
890,20260606,21,620999902,架空カプセルＢ,0000002.000,000000000000,100,W1
891,20260606,50,150999901,架空手術,0000000.000,000000000000,100,W1
892,20260607,90,190999901,架空入院基本料,0000000.000,000000000000,100,W1
893,20260608,90,190999901,架空入院基本料,0000000.000,000000000000,100,W2
894,20260608,21,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000,100,W2
895,20260608,21,620999902,架空カプセルＢ,0000002.000,000000000000,100,W2


How to read these lines:

- `exec_date` is the day of the service. The stay runs from 5 June to 10 June, so there are lines for six days.
- `data_kubun` (データ区分) is the kind of item. 21 is an oral drug, 33 is an injection, 50 is surgery, 60 is a test, 70 is imaging and 90 is the basic inpatient charge (入院基本料).
- `receipt_code` is the code of the item. In real files it comes from the national code masters. Here the codes are invented. In the national masters, drug codes start with 6 and medical procedure codes start with 1. The invented codes follow this.
- `ward_code` is the ward on that day. This patient moves from ward W1 to ward W2 on 8 June. Each hospital chooses its own ward codes.
- `detail_flags` (行為明細区分情報) has 12 digits. Two lines have flags that are not all 0:
  - On 5 June, `002200000000`. The third digit (2) says the patient brought this drug from home and it was not used for the illness that led to the admission. The fourth digit (2) says another hospital prescribed it. The manual sets the points of such a line (`act_points`) to 0.
  - On 10 June, `100000000000`. The first digit (1) says the drug was given at discharge for use at home.

The next table counts the lines in the whole file by kind of item.

In [6]:
from jpomop.spec import load_codes

kubun = load_codes("ef_data_kubun")
t = raw_ef.groupby(["data_kubun", "receipt_code", "item_name"]).size().rename("ef_lines").reset_index()
t.insert(1, "data_kubun_name", t["data_kubun"].map(kubun))
t

,data_kubun,data_kubun_name,receipt_code,item_name,ef_lines
0,21,内服,620999901,架空錠Ａ ５ｍｇ,258
1,21,内服,620999902,架空カプセルＢ,293
2,33,その他（注射）,640999903,架空注射液Ｃ ５００ｍＬ,245
3,50,手術,150999901,架空手術,43
4,60,検査・病理,160999901,架空検査（血液）,145
5,70,画像診断,170999901,架空画像診断,149
6,90,入院基本料,190999901,架空入院基本料,707


## 5. Read and check the files

`read_yoshiki1` and `read_ef` read the files and check the fields that jpomop reads against its specification.
`pivot_yoshiki1` then turns the long Form 1 into one row for each form, one row for each diagnosis and one row for each surgery.

The invented files pass those checks, so there are no reported violations.

In [7]:
from jpomop.parse.ef import read_ef
from jpomop.parse.yoshiki1 import pivot_yoshiki1, read_yoshiki1

y1 = read_yoshiki1(m.yoshiki1)
tables = pivot_yoshiki1(y1.table)
ef = read_ef(m.ef)

print("Form 1 lines:", y1.table.num_rows)
print("forms:", tables.forms.num_rows, " diagnoses:", tables.diagnoses.num_rows,
      " surgeries:", tables.surgeries.num_rows)
print("EF lines:", ef.table.num_rows)
print("violations:", len(y1.violations) + len(tables.violations) + len(ef.violations))

Form 1 lines: 900
forms: 120  diagnoses: 497  surgeries: 43
EF lines: 1840
violations: 0


This is the one form of our patient after the pivot.
The postcode is not kept.

In [8]:
forms = tables.forms.to_pandas()
forms[forms["data_id"] == FOLLOW].T

,56
facility_code,999999999
data_id,0000000048
admission_date,2026-06-05
stay_no,0
summary_no,0
birth_date,1942-09-06
sex,1
admission_route,1
admission_type,100
ambulance,0


The codes for sex, admission route, discharge destination and outcome come from the manual.
jpomop keeps them in small tables.
Our patient is male (1), came from home (1) and went home with follow-up at the same hospital (1). The outcome was "unchanged" (4).

In [9]:
for name in ["y1_sex", "y1_admission_route", "y1_discharge_to", "y1_discharge_outcome"]:
    print(name, load_codes(name))

y1_sex {'1': '男', '2': '女'}
y1_admission_route {'0': '院内の他病棟からの転棟', '1': '家庭からの入院', '4': '他の病院・診療所の病棟からの転院', '5': '介護施設・福祉施設に入所中', '8': '院内で出生', '9': 'その他'}
y1_discharge_to {'0': '院内の他病棟への転棟', '1': '家庭への退院（当院に通院）', '2': '家庭への退院（他の病院・診療所に通院）', '3': '家庭への退院（その他）', '4': '他の病院・診療所への転院', '5': '介護老人保健施設に入所', '6': '介護老人福祉施設に入所', '7': '社会福祉施設、有料老人ホーム等に入所', '8': '終了（死亡等）', '9': 'その他', 'a': '介護医療院', 'b': '未転棟、未退院'}
y1_discharge_outcome {'1': '治癒・軽快', '3': '寛解', '4': '不変', '5': '増悪', '6': '最も医療資源を投入した傷病による死亡', '7': '最も医療資源を投入した傷病以外による死亡', '9': 'その他'}


These are the diagnoses of the same form.
`role` says which line of Form 1 the diagnosis came from.
`suspected` is true when the modifier code 8002 is present.

In [10]:
dx = tables.diagnoses.to_pandas()
dx.loc[dx["data_id"] == FOLLOW,
       ["role", "seq", "icd10", "disease_code", "modifiers", "suspected", "disease_name"]]

,role,seq,icd10,disease_code,modifiers,suspected,disease_name
223,dx_main,0,C189,9000006,NaN,False,結腸癌
224,dx_trigger,0,C189,9000006,NaN,False,結腸癌
225,dx_resource1,0,C189,9000006,NaN,False,結腸癌
226,dx_comorbid,1,E785,9000103,NaN,False,脂質異常症
227,dx_comorbid,2,N189,9000104,NaN,False,慢性腎臓病
228,dx_complication,1,J690,9000201,NaN,False,誤嚥性肺炎


Form 1 also lists the surgery.
jpomop reads these lines, but it does not write them to the CDM.
The procedure table comes from the EF file, which has the same surgery on 6 June (section 8).

In [11]:
sx = tables.surgeries.to_pandas()
sx[sx["data_id"] == FOLLOW]

,facility_code,data_id,admission_date,stay_no,summary_no,seq,surgery_date,surgery_code,surgery_name
22,999999999,0000000048,2026-06-05,0,0,1,2026-06-06,K9999,架空手術


## 6. What a broken file looks like

To see the checks at work, the next cell makes copies of the files with four mistakes:

1. a birth date of 30 February
2. a sex code of 9, which is not in the manual
3. a Form 1 line with 16 fields
4. an EF line with a service date after the discharge date

In [12]:
bad_lines = m.yoshiki1.read_bytes().decode("cp932").splitlines()

a = bad_lines[0].split("\t")   # first line: birth date and sex of patient 0000000001
a[8] = "19520230"               # 30 February
a[9] = "9"                      # sex code that is not in the manual
bad_lines[0] = "\t".join(a)

b = bad_lines[1].split("\t")   # second line: drop the last field
bad_lines[1] = "\t".join(b[:-1])

bad_y1 = work / "yoshiki1_bad.txt"
bad_y1.write_bytes(("\r\n".join(bad_lines) + "\r\n").encode("cp932"))

bad_ef_lines = m.ef.read_bytes().decode("cp932").splitlines()
c = bad_ef_lines[0].split("\t")
c[23] = "20260701"              # service date after discharge
bad_ef_lines[0] = "\t".join(c)
bad_ef = work / "ef_bad.txt"
_ = bad_ef.write_bytes(("\r\n".join(bad_ef_lines) + "\r\n").encode("cp932"))

The checks report each mistake with the file, the line, the field, the value and the rule that was broken.

In [13]:
from jpomop.parse.cross import check_cross_file

r1 = read_yoshiki1(bad_y1)
t1 = pivot_yoshiki1(r1.table)
r2 = read_ef(bad_ef)
found = r1.violations + t1.violations + r2.violations
found += check_cross_file(r2.table, t1.forms, ef_file=bad_ef.name, y1_file=bad_y1.name)

pd.DataFrame([vars(v) for v in found])

,file,line,field,value,rule,message
0,yoshiki1_bad.txt,2,*,16,n_fields,expected 17 fields
1,yoshiki1_bad.txt,1,A000010.1:birth_date,19520230,date_invalid,not a calendar date
2,yoshiki1_bad.txt,1,A000010.2:sex,9,code_not_in_table,not in the code table
3,ef_bad.txt,1,exec_date,2026-07-01,exec_outside_stay,date of service is outside the stay


The same checks run from the command line:

```sh
jpomop validate --yoshiki1 yoshiki1.txt --ef ef.txt --out violations.csv
```

The command ends with status 1 when it finds a violation.
jpomop checks only the fields that it reads.
It is not a replacement for the official checking tool.

## 7. Convert to the OMOP CDM

`run_etl` reads both files and writes the CDM tables into one DuckDB file.
The command line does the same with `jpomop convert`.

The report shows the number of rows in each table.
It also shows what was left out and why.

In [14]:
from jpomop.etl import run_etl

cdm_path = work / "cdm.duckdb"
report = run_etl([m.yoshiki1], [m.ef], cdm_path)

print(pd.Series(report.rows, name="rows").to_string())
print()
print("forms not used:", report.skipped_forms)
print("rows left out:", report.skipped_rows)
print("violations:", report.n_violations, " consistency problems:", report.invariant_problems)

person                  100
observation_period      100
visit_occurrence        120
visit_detail            143
death                     3
condition_occurrence    245
drug_exposure           777
procedure_occurrence    337

forms not used: {}
rows left out: {'condition_occurrence': {'duplicate_in_stay': 216, 'suspected': 36}, 'drug_exposure': {'brought_from_home': 19}}
violations: 0  consistency problems: []


How to read the report:

- 100 patients had 120 stays, so there are 100 rows in `person` and 120 rows in `visit_occurrence`.
- `duplicate_in_stay` is 216. Form 1 often gives the same diagnosis on three lines, as we saw. Without a vocabulary, jpomop writes one row for each ICD-10 code in a stay, not three.
- `suspected` is 36. These are the diagnosis lines with the modifier code 8002. They are not written to `condition_occurrence`.
- `brought_from_home` is 19. These are the EF drug lines for drugs that patients brought from home. They are not written to `drug_exposure`.
- "consistency problems" lists rows that break basic rules, for example an event outside the observation period or a stay that starts after death. The list is empty.

The next sections look at each table for our patient.

## 8. Follow one patient through the tables

### person

One row for each patient.
`person_source_value` is a hash of the hospital code and the patient number, so the original number is not in the output.
`gender_concept_id` 8507 is the OMOP concept for male.
These DPC input files have no race or ethnicity fields, so those concept IDs are 0.

In [15]:
con = duckdb.connect(str(cdm_path), read_only=True)

con.sql(f"""
    SELECT person_id, person_source_value, gender_source_value, gender_concept_id,
           year_of_birth, month_of_birth, day_of_birth
    FROM person WHERE person_id = {FOLLOW_PERSON}
""").df()

,person_id,person_source_value,gender_source_value,gender_concept_id,year_of_birth,month_of_birth,day_of_birth
0,48,5cc23df4ab33c760,1,8507,1942,9,6


### visit_occurrence

One row for each stay.
`visit_concept_id` 9201 is "Inpatient Visit".
`visit_type_concept_id` 32810 is "Claim".
The two source values are the DPC codes: 1 for "came from home" and 1 for "went home, follow-up at this hospital".

In [16]:
con.sql(f"""
    SELECT visit_occurrence_id, visit_concept_id, visit_start_date, visit_end_date,
           visit_type_concept_id, admitted_from_source_value, discharged_to_source_value
    FROM visit_occurrence WHERE person_id = {FOLLOW_PERSON}
""").df()

,visit_occurrence_id,visit_concept_id,visit_start_date,visit_end_date,visit_type_concept_id,admitted_from_source_value,discharged_to_source_value
0,57,9201,2026-06-05,2026-06-10,32810,1,1


### visit_detail

One row for each run of days on the same ward.
Our patient was on ward W1 from 5 to 7 June and on ward W2 from 8 to 10 June.
The second row points back to the first row through `preceding_visit_detail_id`.

In [17]:
con.sql(f"""
    SELECT visit_detail_id, visit_occurrence_id, visit_detail_start_date,
           visit_detail_end_date, visit_detail_source_value, preceding_visit_detail_id
    FROM visit_detail WHERE person_id = {FOLLOW_PERSON} ORDER BY visit_detail_start_date
""").df()

,visit_detail_id,visit_occurrence_id,visit_detail_start_date,visit_detail_end_date,visit_detail_source_value,preceding_visit_detail_id
0,71,57,2026-06-05,2026-06-07,W1,<NA>
1,72,57,2026-06-08,2026-06-10,W2,71


jpomop gets the wards from the EF file, not from Form 1.
This is the ward code on each day of the stay in the EF file.
The two rows above match these ward codes.

In [18]:
(raw_ef[raw_ef["data_id"] == FOLLOW]
 .groupby("exec_date")["ward_code"].agg(lambda s: ",".join(sorted(set(s))))
 .to_frame().T)

exec_date,20260605,20260606,20260607,20260608,20260609,20260610
ward_code,W1,W1,W1,W2,W2,W2


### condition_occurrence

Form 1 gave six diagnosis lines for this patient.
The three lines for C189 become one row, so the table has four rows.

In [19]:
con.sql(f"""
    SELECT condition_occurrence_id, condition_start_date, condition_source_value,
           condition_concept_id, condition_status_source_value, condition_status_concept_id
    FROM condition_occurrence WHERE person_id = {FOLLOW_PERSON}
""").df()

,condition_occurrence_id,condition_start_date,condition_source_value,condition_concept_id,condition_status_source_value,condition_status_concept_id
0,107,2026-06-05,C189,0,dx_main,32902
1,108,2026-06-05,E785,0,dx_comorbid,32907
2,109,2026-06-05,J690,0,dx_complication,32909
3,110,2026-06-05,N189,0,dx_comorbid,32907


Form 1 has no date for each diagnosis, so `condition_start_date` is the admission date for every row.
For J690, which arose after admission, this date is too early.

`condition_status_concept_id` says which line of Form 1 the diagnosis came from:

| Form 1 line | `condition_status_source_value` | `condition_status_concept_id` |
|---|---|---|
| main diagnosis | dx_main | 32902 Primary diagnosis |
| present at admission | dx_comorbid | 32907 Secondary admission diagnosis |
| arose after admission | dx_complication | 32909 Secondary discharge diagnosis |

This choice of concepts is my own reading. It is not an official mapping.

### Suspected diagnoses

Our patient has no suspected diagnosis.
The next cell finds the first patient who has one and shows what happens to it.

In [20]:
SUSPECTED = sorted(dx.loc[dx["suspected"], "data_id"].unique())
print(len(SUSPECTED), "patients have a suspected diagnosis. The first is", SUSPECTED[0])
print(dx.loc[dx["data_id"] == SUSPECTED[0],
             ["admission_date", "role", "icd10", "modifiers", "suspected", "disease_name"]]
      .to_string(index=False))
con.sql(f"""
    SELECT visit_occurrence_id, condition_start_date, condition_source_value,
           condition_status_source_value
    FROM condition_occurrence WHERE person_id = {int(SUSPECTED[0])}
""").df()

12 patients have a suspected diagnosis. The first is 0000000009
admission_date         role icd10 modifiers  suspected disease_name
    2026-06-06      dx_main  I500       NaN      False      うっ血性心不全
    2026-06-06   dx_trigger  I500       NaN      False      うっ血性心不全
    2026-06-06 dx_resource1  I500       NaN      False      うっ血性心不全
    2026-06-06  dx_comorbid   I10       NaN      False      本態性高血圧症
    2026-06-13      dx_main  J189      8002       True        肺炎の疑い
    2026-06-13   dx_trigger  J189      8002       True        肺炎の疑い
    2026-06-13 dx_resource1  J189      8002       True        肺炎の疑い


,visit_occurrence_id,condition_start_date,condition_source_value,condition_status_source_value
0,11,2026-06-06,I10,dx_comorbid
1,11,2026-06-06,I500,dx_main


Patient 0000000009 has two stays.
In the second stay the main diagnosis is J189 with modifier 8002, which is suspected pneumonia.
The CDM table has rows only for the first stay; the second stay has no condition row at all.
jpomop leaves it out so it is not counted as a confirmed disease.
It also means that any count that starts from `condition_occurrence` misses this stay (see section 12).

### drug_exposure

These are the EF drug lines of our patient.

In [21]:
is_drug = raw_ef["receipt_code"].str.startswith("6")
raw_ef.loc[(raw_ef["data_id"] == FOLLOW) & is_drug,
           ["exec_date", "receipt_code", "item_name", "quantity", "detail_flags"]].sort_values("exec_date", kind="stable")

,exec_date,receipt_code,item_name,quantity,detail_flags
902,20260605,620999901,架空錠Ａ ５ｍｇ,0000001.000,002200000000
890,20260606,620999902,架空カプセルＢ,0000002.000,000000000000
894,20260608,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000
895,20260608,620999902,架空カプセルＢ,0000002.000,000000000000
897,20260609,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000
898,20260609,620999901,架空錠Ａ ５ｍｇ,0000001.000,000000000000
901,20260610,640999903,架空注射液Ｃ ５００ｍＬ,0000001.000,000000000000
903,20260610,620999902,架空カプセルＢ,0000002.000,100000000000


There are eight lines, but the table below has seven rows.
The line of 5 June is missing because the patient brought that drug from home.
The line of 10 June has `stop_reason` "discharge_prescription" because the drug was given at discharge.

In [22]:
con.sql(f"""
    SELECT drug_exposure_id, drug_exposure_start_date, drug_source_value, drug_concept_id,
           quantity, stop_reason, route_source_value, dose_unit_source_value
    FROM drug_exposure WHERE person_id = {FOLLOW_PERSON}
    ORDER BY drug_exposure_start_date, drug_exposure_id
""").df()

,drug_exposure_id,drug_exposure_start_date,drug_source_value,drug_concept_id,quantity,stop_reason,route_source_value,dose_unit_source_value
0,378,2026-06-06,620999902,0,2.0,NaN,21,009
1,379,2026-06-08,620999901,0,1.0,NaN,21,009
2,380,2026-06-08,620999902,0,2.0,NaN,21,009
3,381,2026-06-09,640999903,0,1.0,NaN,33,033
4,382,2026-06-09,620999901,0,1.0,NaN,21,009
5,383,2026-06-10,640999903,0,1.0,NaN,33,033
6,384,2026-06-10,620999902,0,2.0,discharge_prescription,21,009


The start date is the service date in the EF file.
`quantity` comes from the EF file.
`route_source_value` keeps the kind of item (21 oral, 33 injection), and `dose_unit_source_value` keeps the unit code.

`drug_concept_id` is 0 because no vocabulary was given.
Section 11 shows the conversion with a vocabulary.

### procedure_occurrence

One row for each EF line whose code starts with 1, except room charges and meals.
For this patient these are two imaging lines (5 and 9 June) and the surgery (6 June).
The six room-charge lines are not in the table.

`procedure_concept_id` is always 0 in this version.
jpomop does not map Japanese procedure codes to standard concepts yet.

In [23]:
con.sql(f"""
    SELECT procedure_occurrence_id, procedure_date, procedure_source_value,
           procedure_concept_id, visit_occurrence_id
    FROM procedure_occurrence WHERE person_id = {FOLLOW_PERSON} ORDER BY procedure_date
""").df()

,procedure_occurrence_id,procedure_date,procedure_source_value,procedure_concept_id,visit_occurrence_id
0,167,2026-06-05,170999901,0,57
1,168,2026-06-06,150999901,0,57
2,169,2026-06-09,170999901,0,57


### observation_period

One row for each patient, from the first admission to the last discharge in the files.
Our patient has one stay, so the next cell shows the first patient with two stays instead.
The period covers both stays.

These inpatient DPC files do not show whether a patient is observed between two stays.
Keep this in mind for any study that counts time at risk.

In [24]:
two_stays = con.sql("""
    SELECT min(person_id) FROM (SELECT person_id FROM visit_occurrence
                                GROUP BY 1 HAVING count(*) > 1)
""").fetchone()[0]
con.sql(f"""
    SELECT o.person_id, o.observation_period_start_date, o.observation_period_end_date,
           v.visit_occurrence_id, v.visit_start_date, v.visit_end_date
    FROM observation_period o JOIN visit_occurrence v USING (person_id)
    WHERE o.person_id = {two_stays} ORDER BY v.visit_start_date
""").df()

,person_id,observation_period_start_date,observation_period_end_date,visit_occurrence_id,visit_start_date,visit_end_date
0,6,2026-06-09,2026-06-25,6,2026-06-09,2026-06-14
1,6,2026-06-09,2026-06-25,7,2026-06-19,2026-06-25


## 9. Deaths

The discharge outcome in these Form 1 files records deaths in hospital, not deaths after discharge.
jpomop writes a `death` row when the discharge outcome in Form 1 is 6 or 7.
The death date is the discharge date.

In [25]:
con.sql("""
    SELECT d.person_id, d.death_date, d.death_type_concept_id,
           v.visit_start_date, v.visit_end_date, v.discharged_to_source_value
    FROM death d JOIN visit_occurrence v
      ON v.person_id = d.person_id AND v.visit_end_date = d.death_date
    ORDER BY d.person_id
""").df()

,person_id,death_date,death_type_concept_id,visit_start_date,visit_end_date,discharged_to_source_value
0,25,2026-06-14,32810,2026-06-11,2026-06-14,8
1,56,2026-06-15,32810,2026-06-11,2026-06-15,8
2,70,2026-06-26,32810,2026-06-18,2026-06-26,8


These are the Form 1 rows behind the three deaths.
All three have discharge destination 8 ("ended, including death") and outcome 7.
Outcome 6 means death from the diagnosis that used the most resources.
Outcome 7 means death from another cause.
Deaths after discharge are not recorded in these Form 1 files, so they are not in the `death` table.

In [26]:
forms.loc[forms["discharge_outcome"].isin(["6", "7"]),
          ["data_id", "admission_date", "discharge_date", "discharge_to", "discharge_outcome"]]

,data_id,admission_date,discharge_date,discharge_to,discharge_outcome
30,0000000025,2026-06-11,2026-06-14,8,7
66,0000000056,2026-06-11,2026-06-15,8,7
84,0000000070,2026-06-18,2026-06-26,8,7


## 10. Forms that are counted but not converted

A hospital can write more than one Form 1 for one stay.
For example, when a patient moves between ward groups, the hospital writes one form for the whole stay and one form for each ward group.
The whole-stay form has `summary_no` 0.
The ward-group forms have 1, 2 and so on.

The invented data has only whole-stay forms.
So the next cell copies the form of our patient, changes `summary_no` to 1 and adds it to the file.
This is a rough stand-in for a ward-group form.

In [27]:
src = m.yoshiki1.read_bytes().decode("cp932").splitlines()
extra = []
for ln in src:
    f = ln.split("\t")
    if f[1] == FOLLOW:
        f[4] = "1"              # 統括診療情報番号: 0 -> 1 (ward-group form)
        extra.append("\t".join(f))

y1_more = work / "yoshiki1_with_ward_group_form.txt"
y1_more.write_bytes(("\r\n".join(src + extra) + "\r\n").encode("cp932"))

report2 = run_etl([y1_more], [m.ef], work / "cdm2.duckdb")
print("forms not used:", report2.skipped_forms)
print("visit_occurrence rows:", report2.rows["visit_occurrence"],
      " condition_occurrence rows:", report2.rows["condition_occurrence"])

forms not used: {'summary_no=1': 1}
visit_occurrence rows: 120  condition_occurrence rows: 245


The extra form is counted under "forms not used".
The numbers of stays and diagnoses are the same as before (120 and 245), so the stay was not counted twice.

jpomop treats the forms for joined readmissions (A and B) in the same way.
It uses the day-90 interim form (Z) only when a stay has no whole-stay form yet.
I have not checked these rules on real files.

## 11. Concept IDs and the vocabulary

So far every concept ID for a diagnosis, drug or procedure is 0.
The original codes are in the source value columns.

In [28]:
con.sql("""
    SELECT 'condition_occurrence' AS cdm_table, count(*) AS n_rows,
           count(*) FILTER (WHERE condition_concept_id = 0) AS concept_id_0
    FROM condition_occurrence
    UNION ALL SELECT 'drug_exposure', count(*),
           count(*) FILTER (WHERE drug_concept_id = 0) FROM drug_exposure
    UNION ALL SELECT 'procedure_occurrence', count(*),
           count(*) FILTER (WHERE procedure_concept_id = 0) FROM procedure_occurrence
""").df()

,cdm_table,n_rows,concept_id_0
0,condition_occurrence,245,245
1,drug_exposure,777,777
2,procedure_occurrence,337,337


To fill in the concept IDs, jpomop needs three files that you download yourself:

- the Athena vocabularies (ICD10, SNOMED, RxNorm, RxNorm Extension)
- the drug master (医薬品マスター), which links each drug's receipt code to its NHI drug price code
- the table by Kimura et al. (2025), which links NHI drug price codes to OMOP concepts

The full Athena download is large and needs a free account, so this notebook does not include it.
Instead, the next cell holds a small extract for the 12 ICD-10 codes in the invented data.
These rows are real: they were copied from the OMOP standard vocabularies on 14 October 2026.
For real work, download the full vocabularies from https://athena.ohdsi.org.

Each ICD-10 code (a non-standard concept) has a "Maps to" link to a standard SNOMED concept.
J69.0 links to two standard concepts.

In [29]:
# Rows copied from the OMOP standard vocabularies on 14 October 2026.
# (concept_id, concept_code, vocabulary_id, concept_class_id, standard_concept, concept_name)
CONCEPTS = [
    (45561790, 'C18.9', 'ICD10', 'ICD10 code', '', 'Malignant neoplasm: Colon, unspecified'),
    (197500, '93761005', 'SNOMED', 'Disorder', 'S', 'Primary malignant neoplasm of colon'),
    (45561952, 'E11.9', 'ICD10', 'ICD10 code', '', 'Type 2 diabetes mellitus, without complications'),
    (201826, '44054006', 'SNOMED', 'Disorder', 'S', 'Type 2 diabetes mellitus'),
    (45537986, 'E78.5', 'ICD10', 'ICD10 code', '', 'Hyperlipidaemia, unspecified'),
    (432867, '55822004', 'SNOMED', 'Disorder', 'S', 'Hyperlipidemia'),
    (45591453, 'I10', 'ICD10', 'ICD10 Hierarchy', '', 'Essential (primary) hypertension'),
    (320128, '59621000', 'SNOMED', 'Disorder', 'S', 'Essential hypertension'),
    (45567179, 'I50.0', 'ICD10', 'ICD10 code', '', 'Congestive heart failure'),
    (319835, '42343007', 'SNOMED', 'Disorder', 'S', 'Congestive heart failure'),
    (45596216, 'I63.9', 'ICD10', 'ICD10 code', '', 'Cerebral infarction, unspecified'),
    (443454, '432504007', 'SNOMED', 'Disorder', 'S', 'Cerebral infarction'),
    (45601123, 'J18.9', 'ICD10', 'ICD10 code', '', 'Pneumonia, unspecified'),
    (255848, '233604007', 'SNOMED', 'Disorder', 'S', 'Pneumonia'),
    (45591562, 'J69.0', 'ICD10', 'ICD10 code', '', 'Pneumonitis due to food and vomit'),
    (4112839, '196033004', 'SNOMED', 'Disorder', 'S', 'Pneumonitis caused by inhalation of regurgitated food'),
    (4110180, '196035006', 'SNOMED', 'Disorder', 'S', 'Pneumonitis caused by inhalation of vomitus'),
    (45552951, 'K80.2', 'ICD10', 'ICD10 code', '', 'Calculus of gallbladder without cholecystitis'),
    (36716712, '722869007', 'SNOMED', 'Disorder', 'S', 'Calculus of gallbladder without cholecystitis or cholangitis'),
    (45553438, 'N18.9', 'ICD10', 'ICD10 code', '', 'Chronic kidney disease, unspecified'),
    (46271022, '709044004', 'SNOMED', 'Disorder', 'S', 'Chronic kidney disease'),
    (45548665, 'N39.0', 'ICD10', 'ICD10 code', '', 'Urinary tract infection, site not specified'),
    (81902, '68566005', 'SNOMED', 'Disorder', 'S', 'Urinary tract infectious disease'),
    (1529272, 'S72.00', 'ICD10', 'ICD10 code', '', 'Fracture of neck of femur, closed'),
    (434500, '359820003', 'SNOMED', 'Disorder', 'S', 'Closed fracture of neck of femur'),
]
# (ICD10 concept_id, standard concept_id) pairs with the relationship "Maps to"
MAPS_TO = [
    (45561790, 197500), (45561952, 201826), (45537986, 432867),
    (45591453, 320128), (45567179, 319835), (45596216, 443454),
    (45601123, 255848), (45591562, 4112839), (45591562, 4110180),
    (45552951, 36716712), (45553438, 46271022), (45548665, 81902),
    (1529272, 434500),
]

athena = work / "athena_extract"
athena.mkdir()
concept_lines = ["concept_id\tconcept_name\tdomain_id\tvocabulary_id\tconcept_class_id\t"
                 "standard_concept\tconcept_code\tvalid_start_date\tvalid_end_date\tinvalid_reason"]
for cid, code, vocabulary, cls, standard, name in CONCEPTS:
    concept_lines.append(f"{cid}\t{name}\tCondition\t{vocabulary}\t{cls}\t{standard}\t{code}"
                         "\t19700101\t20991231\t")
(athena / "CONCEPT.csv").write_text("\n".join(concept_lines) + "\n", encoding="utf-8")
relationship_lines = ["concept_id_1\tconcept_id_2\trelationship_id\tvalid_start_date\t"
                      "valid_end_date\tinvalid_reason"]
for source, target in MAPS_TO:
    relationship_lines.append(f"{source}\t{target}\tMaps to\t19700101\t20991231\t")
_ = (athena / "CONCEPT_RELATIONSHIP.csv").write_text("\n".join(relationship_lines) + "\n",
                                                     encoding="utf-8")

pd.DataFrame(CONCEPTS, columns=["concept_id", "concept_code", "vocabulary_id",
                                "concept_class_id", "standard_concept", "concept_name"])

,concept_id,concept_code,vocabulary_id,concept_class_id,standard_concept,concept_name
0,45561790,C18.9,ICD10,ICD10 code,,"Malignant neoplasm: Colon, unspecified"
1,197500,93761005,SNOMED,Disorder,S,Primary malignant neoplasm of colon
2,45561952,E11.9,ICD10,ICD10 code,,"Type 2 diabetes mellitus, without complications"
3,201826,44054006,SNOMED,Disorder,S,Type 2 diabetes mellitus
4,45537986,E78.5,ICD10,ICD10 code,,"Hyperlipidaemia, unspecified"
5,432867,55822004,SNOMED,Disorder,S,Hyperlipidemia
6,45591453,I10,ICD10,ICD10 Hierarchy,,Essential (primary) hypertension
7,320128,59621000,SNOMED,Disorder,S,Essential hypertension
8,45567179,I50.0,ICD10,ICD10 code,,Congestive heart failure
9,319835,42343007,SNOMED,Disorder,S,Congestive heart failure


The drugs are different.
The receipt codes in the invented data are made up, so they cannot match the real drug master or the real Kimura table.
The next cell writes two small invented files of the same shape instead.
The concept IDs in them start with 9000 and do not exist in the OMOP vocabularies.
The third drug has no match on purpose, to show what happens to a code with no match.

In [30]:
# A file shaped like the drug master: 42 fields, receipt code in field 3,
# NHI drug price code in field 32. Both codes are invented.
pairs = {"620999901": "9999001F1011", "620999902": "9999002M1011", "640999903": "9999400A1011"}
rows = []
for receipt, price_code in pairs.items():
    fields = ["0", "Y", receipt] + [""] * 39
    fields[31] = price_code
    rows.append(",".join(f'"{x}"' for x in fields))
drug_master = work / "y_invented.csv"
drug_master.write_bytes(("\r\n".join(rows) + "\r\n").encode("cp932"))

# A file shaped like rxnorm_mapping.tsv. The concept IDs are invented (9000...).
# The third drug has no match on purpose.
mapping = work / "rxnorm_mapping_invented.tsv"
_ = mapping.write_text("Category\tNHIDPLC \tOSV_ID\tOSV_NAME\n"
                       "PO\t9999001F1011\t900001001\tinvented drug A\n"
                       "PO\t9999002M1011\t900001002\tinvented drug B\n"
                       "Injection\t9999400A1011\tN/A\tN/A\n", encoding="utf-8")

`build_vocab` joins the three files into one small DuckDB file.

DPC writes ICD-10 codes without a dot and sometimes with a fifth character.
jpomop tries the most specific code first and then shorter ones.
S7200 is found as S72.00, "Fracture of neck of femur, closed".
J690 gives two standard concepts.

In [31]:
from jpomop.vocab import Vocab, build_vocab, icd10_candidates

vocab_path = work / "vocab.duckdb"
vr = build_vocab(athena, [drug_master], mapping, vocab_path)
print("ICD-10 codes in the extract:", vr.icd10_codes, " with a standard concept:", vr.icd10_mapped)
print("drugs in the drug master:", vr.receipt_drugs, " with a concept:", vr.receipt_drugs_mapped)
print()

v = Vocab.load(vocab_path)
for code in ["C189", "S7200", "J690"]:
    hits = v.map_icd10_all(code)
    print(code, "tries", icd10_candidates(code), "-> ICD10 concept", hits[0].source_concept_id,
          "-> standard concept(s)", [h.concept_id for h in hits])
for code in ["620999901", "640999903"]:
    print(code, "-> drug concept", v.map_drug(code).concept_id)

ICD-10 codes in the extract: 12  with a standard concept: 12
drugs in the drug master: 3  with a concept: 2

C189 tries ['C18.9', 'C18'] -> ICD10 concept 45561790 -> standard concept(s) [197500]
S7200 tries ['S72.00', 'S72.0', 'S72'] -> ICD10 concept 1529272 -> standard concept(s) [434500]
J690 tries ['J69.0', 'J69'] -> ICD10 concept 45591562 -> standard concept(s) [4110180, 4112839]
620999901 -> drug concept 900001001
640999903 -> drug concept 0


Now the same conversion with the vocabulary.
`condition_source_concept_id` is the ICD-10 concept.
`condition_concept_id` is the standard concept that it maps to.

J690 now has two rows for each stay, one for each standard concept.
This follows the OMOP conventions for a source code that maps to more than one concept.
So `condition_occurrence` has 259 rows here, 14 more than without the vocabulary (245).

In [32]:
cdm3 = work / "cdm_with_vocab.duckdb"
report3 = run_etl([m.yoshiki1], [m.ef], cdm3, vocab=vocab_path)

con3 = duckdb.connect(str(cdm3), read_only=True)
con3.sql(f"""
    SELECT c.condition_source_value, c.condition_source_concept_id, c.condition_concept_id,
           n.concept_name AS standard_concept_name, count(*) AS n_rows
    FROM condition_occurrence c
    LEFT JOIN read_csv('{athena / "CONCEPT.csv"}', delim='\t', header=true, quote='',
                       all_varchar=true) n
      ON n.concept_id = CAST(c.condition_concept_id AS VARCHAR)
    GROUP BY ALL ORDER BY c.condition_source_value
""").df()

,condition_source_value,condition_source_concept_id,condition_concept_id,standard_concept_name,n_rows
0,C189,45561790,197500,Primary malignant neoplasm of colon,12
1,E119,45561952,201826,Type 2 diabetes mellitus,27
2,E785,45537986,432867,Hyperlipidemia,28
3,I10,45591453,320128,Essential hypertension,35
4,I500,45567179,319835,Congestive heart failure,15
5,I639,45596216,443454,Cerebral infarction,17
6,J189,45601123,255848,Pneumonia,12
7,J690,45591562,4112839,Pneumonitis caused by inhalation of regurgitated food,14
8,J690,45591562,4110180,Pneumonitis caused by inhalation of vomitus,14
9,K802,45552951,36716712,Calculus of gallbladder without cholecystitis or cholangitis,18


`coverage` counts how many rows and how many different codes got a nonzero concept ID.
It also lists the codes that did not, with the most frequent first.
The command line shows the same with `jpomop report`.

With the full vocabularies this table shows which codes most need mapping.
The diagnosis row is 100% here only because the extract was made for exactly these 12 codes.
The drug row only reflects the invented drug files.
Neither number says anything about real data.
Procedures stay at 0% because jpomop has no procedure mapping yet.

In [33]:
from jpomop.report import coverage

pd.DataFrame(
    [(c.table, c.rows, c.mapped_rows, f"{c.row_rate:.0%}", c.codes, c.mapped_codes, c.top_unmapped)
     for c in coverage(cdm3)],
    columns=["cdm_table", "rows", "mapped_rows", "row_rate", "codes", "mapped_codes", "unmapped_codes"],
)

,cdm_table,rows,mapped_rows,row_rate,codes,mapped_codes,unmapped_codes
0,condition_occurrence,259,259,100%,12,12,[]
1,drug_exposure,777,532,68%,3,2,"[(640999903, 245)]"
2,procedure_occurrence,337,0,0%,3,0,"[(170999901, 149), (160999901, 145), (150999901, 43)]"


## 12. Four questions asked of the tables

The tables are in a DuckDB file, so plain SQL works.
Because the data is invented, the answers are only examples of how the tables join.

### How long are the stays for each main diagnosis?

`condition_status_concept_id` 32902 marks the main diagnosis.

In [34]:
con.sql("""
    SELECT c.condition_source_value AS main_diagnosis_icd10,
           count(*) AS stays,
           round(avg(v.visit_end_date - v.visit_start_date + 1), 1) AS mean_days,
           min(v.visit_end_date - v.visit_start_date + 1) AS min_days,
           max(v.visit_end_date - v.visit_start_date + 1) AS max_days
    FROM visit_occurrence v
    JOIN condition_occurrence c USING (visit_occurrence_id)
    WHERE c.condition_status_concept_id = 32902
    GROUP BY 1 ORDER BY stays DESC
""").df()

,main_diagnosis_icd10,stays,mean_days,min_days,max_days
0,S7200,19,6.4,2,10
1,K802,18,5.1,2,10
2,I639,17,6.1,2,10
3,N390,15,5.8,2,10
4,I500,15,5.0,2,9
5,C189,12,6.1,2,10
6,J189,12,7.4,4,10


The stays add up to 108, not 120.
The other 12 stays have a suspected main diagnosis, so they have no main diagnosis row.
This is a direct result of leaving suspected diagnoses out.
A study that needs every stay should start from `visit_occurrence`, not from the diagnosis.

### How many patients got each drug, and how many lines were given at discharge?

In [35]:
con.sql("""
    SELECT d.drug_source_value,
           count(DISTINCT d.person_id) AS persons,
           count(DISTINCT d.visit_occurrence_id) AS stays,
           count(*) AS rows_,
           count(*) FILTER (WHERE d.stop_reason = 'discharge_prescription') AS at_discharge
    FROM drug_exposure d GROUP BY 1 ORDER BY 1
""").df()

,drug_source_value,persons,stays,rows_,at_discharge
0,620999901,87,103,239,0
1,620999902,98,115,293,46
2,640999903,94,109,245,0


### How many stays include a ward change?

In [36]:
con.sql("""
    SELECT wards_in_stay, count(*) AS stays
    FROM (SELECT visit_occurrence_id, count(*) AS wards_in_stay
          FROM visit_detail GROUP BY 1)
    GROUP BY 1 ORDER BY 1
""").df()

,wards_in_stay,stays
0,1,97
1,2,23


### Which patients were admitted more than once, and how many days after the last discharge?

In [37]:
con.sql("""
    SELECT person_id, visit_occurrence_id, visit_start_date, visit_end_date,
           visit_start_date - lag(visit_end_date) OVER (PARTITION BY person_id
                                                         ORDER BY visit_start_date) AS days_since_last_discharge
    FROM visit_occurrence
    QUALIFY count(*) OVER (PARTITION BY person_id) > 1
    ORDER BY person_id, visit_start_date
    LIMIT 8
""").df()

,person_id,visit_occurrence_id,visit_start_date,visit_end_date,days_since_last_discharge
0,6,6,2026-06-09,2026-06-14,<NA>
1,6,7,2026-06-19,2026-06-25,5
2,7,8,2026-06-07,2026-06-14,<NA>
3,7,9,2026-06-16,2026-06-19,2
4,9,11,2026-06-06,2026-06-09,<NA>
5,9,12,2026-06-13,2026-06-15,4
6,14,17,2026-06-10,2026-06-16,<NA>
7,14,18,2026-06-19,2026-06-28,3


Patients 7, 9 and 14 came back within 7 days.
In real files, a hospital may also write a joined form (A) for such a readmission, when the conditions in the manual are met.
The invented data does not make joined forms.

## 13. Parquet output

The same tables can be written as Parquet files, one file for each table.

In [38]:
report4 = run_etl([m.yoshiki1], [m.ef], work / "cdm_parquet", fmt="parquet")
sorted(p.name for p in (work / "cdm_parquet").iterdir())

['condition_occurrence.parquet',
 'death.parquet',
 'drug_exposure.parquet',
 'observation_period.parquet',
 'person.parquet',
 'procedure_occurrence.parquet',
 'visit_detail.parquet',
 'visit_occurrence.parquet']

## 14. What this notebook does not show

- It does not show how jpomop works on real hospital files. I have not run it on real data.
- It does not show real mapping rates. The diagnosis extract in section 11 covers only the 12 codes of the invented data, and the drug files are invented.
- Procedures have no standard concepts yet.
- Costs and points are not converted.
- Outpatient files, the D file and the H file are not read.
- The handling of ward-group forms, readmission forms and interim forms follows my reading of the manual. It needs checking on real files.

If you work with DPC data and can run jpomop where your data is, counts and error messages would help.
You do not need to share any data.
Issues are open at https://github.com/kojikojikojikoji/jpomop/issues

The cell below closes the files and deletes the temporary folder.

In [39]:
con.close()
con3.close()
import shutil

shutil.rmtree(work)